# 10 - accDM daughter: momentum/multipole resolution study (the safe speedup)

The exact hierarchy is the production path; its cost is dominated by the daughter perturbation vector of size `(l_max_ncdm+1) x q_size` = `18 x 1001 ~ 18000` ODE variables. The fluid (architecturally unstable, ~1.8x ceiling) and parent-slaving (daughter free-streams away) are both dead ends. This notebook asks the untested question: **does the daughter actually need q_size = 1001 and l_max_ncdm = 17?**

Method: take the full-resolution exact run as the convergence reference, then sweep (1) the daughter momentum bins `q_size` (SAFE - daughter only) and (2) the global `l_max_ncdm` (COUPLED - also affects the neutrino, so we watch the high-l C_l tail). Find the smallest grid whose P(k), C_l and background w stay within tolerance of the reference - a pure-config speedup with the exact hierarchy and physics untouched. No rebuild: runs against the built classy. See docs/superpowers/specs/2026-06-24-daughter-momentum-resolution-study.md.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

plt.rcParams.update({'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 300})
qual = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

def find_key(d, *cands):
    for c in cands:
        if c in d:
            return c
    for key in d:
        for c in cands:
            if c in key:
                return key
    raise KeyError('none of {} in keys like {}'.format(cands, list(d)[:20]))

## Parameters and run helper (same boosted model as notebook 6, kappa = 4.0)

In [ ]:
omega_b = 0.022383; omega_cdm0 = 0.12011\nA_s = 2.1005829616811546e-9; n_s = 0.96605; tau_reio = 0.0543; H0 = 67.32\nA_T = 0.13; MASS = 1e16; F_ACC = 0.1; ETA = 0.1; A_REC = 1.0 / (1.0 + 1090.0)\nKAPPA = 4.0\nLMAX = 2000                         # multipole at which C_l deviations are evaluated\nK_NODES = np.logspace(-3, 0.9, 50)  # 1/Mpc\n\nTOL_PK = 1e-3; TOL_CL = 1e-3; TOL_TAIL = 1e-3\n\ndef base_exact(n_bins_d, l_max):\n    ocdm = omega_cdm0 * (1 + F_ACC*(1 - A_REC**KAPPA)/(1 + (A_REC/A_T)**KAPPA))**(-1)\n    p = {'omega_b': omega_b, 'omega_cdm': ocdm, 'H0': H0,\n         'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}\n    p.update({'output': 'tCl,mPk', 'l_max_scalars': 2500, 'P_k_max_1/Mpc': 10.0,\n              'z_max_pk': 0.0, 'reionization_z_start_max': 80,\n              'evolver': 0, 'background_Nloga': 5000, 'gauge': 'synchronous',\n              'l_max_ncdm': int(l_max),\n              'vary_Gamma_acc': 'yes', 'kappa_acc': KAPPA, 'a_t_acc': A_T,\n              'f_acc': F_ACC, 'eta_acc': ETA,\n              'm_acc_in_GeV': MASS, 'm_cdm_in_GeV': MASS,\n              'N_ncdm': 2, 'deg_ncdm': '3, 1',\n              'm_ncdm': '0.02, {:.6e}'.format(MASS*1e9),\n              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',\n              'ncdm_N_momentum_bins': '15, {:d}'.format(int(n_bins_d)),\n              'N_ur': 0.00441, 'ncdm_fluid_approximation': 3})\n    return p\n\ndef run_case(n_bins_d, l_max):\n    M = Class(); M.set(base_exact(n_bins_d, l_max))\n    t0 = time.perf_counter(); M.compute(); dt = time.perf_counter() - t0\n    pk = np.array([M.pk(float(k), 0.0) for k in K_NODES])\n    cl = M.raw_cl(LMAX)\n    bg = M.get_background()\n    a_bg = 1.0 / (1.0 + bg[find_key(bg, 'z')])\n    rho = bg[find_key(bg, '(.)rho_ncdm[1]', 'rho_ncdm[1]')]\n    pres = bg[find_key(bg, '(.)p_ncdm[1]', 'p_ncdm[1]')]\n    w = pres / rho\n    o = np.argsort(a_bg); a_bg = a_bg[o]; w = w[o]\n    M.struct_cleanup(); M.empty()\n    return dict(n_bins=int(n_bins_d), l_max=int(l_max), t=dt, pk=pk,\n                ell=np.asarray(cl['ell']), tt=np.asarray(cl['tt']), a_bg=a_bg, w=w)\n\ndef dev_pk(c, ref):\n    return float(np.max(np.abs(c['pk'] / ref['pk'] - 1.0)))\n\ndef dev_cl(c, ref, ell_lo=2, ell_hi=None):\n    e = ref['ell']; m = e >= ell_lo\n    if ell_hi is not None: m = m & (e <= ell_hi)\n    r = ref['tt'][m]; cc = c['tt'][m]\n    s = np.where(np.abs(r) > 0, np.abs(r), 1.0)\n    return float(np.max(np.abs(cc - r) / s))\n\ndef dev_w(c, ref, w_floor=1e-2):\n    # DIAGNOSTIC only (not a gate). Compare where the reference w is non-negligible: the\n    # daughter goes cold (w -> ~0) at late a, where a relative metric explodes on a\n    # physically irrelevant absolute difference. P(k) already subsumes any real background\n    # shift, so we gate on observables and only report dW.\n    mask = np.abs(ref['w']) > w_floor\n    if not np.any(mask): return float('nan')\n    ag = ref['a_bg'][mask]; wr = ref['w'][mask]\n    wc = np.interp(ag, c['a_bg'], c['w'])\n    rel = np.abs(wc - wr) / np.abs(wr)\n    rel = rel[np.isfinite(rel)]\n    return float(np.max(rel)) if rel.size else float('nan')

## Reference: full-resolution exact run (q_size = 1001, l_max_ncdm = 17)

In [ ]:
REF_NBINS = 1001; REF_LMAX = 17
ref = run_case(REF_NBINS, REF_LMAX)
print('reference (q_size={}, l_max={}): {:.1f}s'.format(REF_NBINS, REF_LMAX, ref['t']))
print('background w_ncdm[1]: at a=0.1 -> {:.3e}, at a=1 -> {:.3e}'.format(
    float(np.interp(0.1, ref['a_bg'], ref['w'])), float(np.interp(1.0, ref['a_bg'], ref['w']))))

## Sweep 1 (primary, SAFE): daughter momentum bins q_size, l_max fixed at 17

Deviations are vs the full-resolution reference. We also track background w_ncdm[1](a) because the manual quadrature ties the background grid to the same input (background.c:1594), so a coarse q_size could shift the background PSD.

In [ ]:
NBINS_SWEEP = [1001, 500, 250, 120, 60, 30]
qrows = []
for nb in NBINS_SWEEP:
    c = ref if (nb == REF_NBINS) else run_case(nb, REF_LMAX)
    row = dict(n_bins=nb, t=c['t'], speedup=ref['t'] / c['t'],
               dpk=dev_pk(c, ref), dcl=dev_cl(c, ref),
               dtail=dev_cl(c, ref, ell_lo=1500), dw=dev_w(c, ref))
    qrows.append(row)
    print('q_size={:>5}  {:6.1f}s  {:5.2f}x  dPk={:.2e}  dCl={:.2e}  dTail={:.2e}  dW={:.2e}'.format(
        nb, row['t'], row['speedup'], row['dpk'], row['dcl'], row['dtail'], row['dw']))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
nb = [r['n_bins'] for r in qrows]
ax[0].loglog(nb, [r['dpk'] for r in qrows], 'o-', color=qual[0], label='P(k)')
ax[0].loglog(nb, [r['dcl'] for r in qrows], 's-', color=qual[1], label='C_l TT')
ax[0].loglog(nb, [r['dtail'] for r in qrows], '^-', color=qual[2], label='C_l tail (l>=1500)')
ax[0].loglog(nb, [r['dw'] for r in qrows], 'd-', color=qual[3], label='background w')
ax[0].axhline(TOL_PK, color='k', ls='--', lw=1)
ax[0].set_xlabel('daughter momentum bins q_size'); ax[0].set_ylabel('max |deviation| vs full-res')
ax[0].set_title('convergence vs q_size'); ax[0].legend(loc='best'); ax[0].grid(True, which='both', alpha=0.3)
ax[1].semilogx(nb, [r['t'] for r in qrows], 'o-', color=qual[0])
ax[1].set_xlabel('daughter momentum bins q_size'); ax[1].set_ylabel('wall-time [s]')
ax[1].set_title('cost vs q_size'); ax[1].grid(True, which='both', alpha=0.3)
ax[2].semilogx(nb, [r['speedup'] for r in qrows], 'o-', color=qual[2])
ax[2].axhline(1.0, color='k', ls=':', lw=1)
ax[2].set_xlabel('daughter momentum bins q_size'); ax[2].set_ylabel('speedup (t_ref / t)')
ax[2].set_title('speedup vs q_size'); ax[2].grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
# Gate on the OBSERVABLES (P(k), C_l, tail). dW is a DIAGNOSTIC only: its large values are a\n# cold-tail artifact (w->0 at late a inflates a relative metric on a negligible absolute\n# difference), and P(k) already captures any real background shift. (Earlier the dW gate\n# wrongly rejected every reduced q_size -> picked 1001.)\nprint('q_size recommendation vs tolerance (gate: P(k), C_l, tail):')\nfor tol in [1e-3, 3e-3, 1e-2]:\n    cand = [r for r in qrows if r['dpk'] < tol and r['dcl'] < tol and r['dtail'] < tol]\n    nbc = min((r['n_bins'] for r in cand), default=REF_NBINS)\n    sp = ref['t'] / next(r['t'] for r in qrows if r['n_bins'] == nbc)\n    print('  tol={:.0e}:  q_size={:>5}   speedup={:5.1f}x'.format(tol, nbc, sp))\nGATE = 1e-2\nCHOSEN_NBINS = min((r['n_bins'] for r in qrows\n                    if r['dpk'] < GATE and r['dcl'] < GATE and r['dtail'] < GATE), default=REF_NBINS)\nt_chosen = next(r['t'] for r in qrows if r['n_bins'] == CHOSEN_NBINS)\nprint('using CHOSEN_NBINS={} (gate {:.0e} on observables) for the l_max sweep -> {:.1f}x'.format(\n    CHOSEN_NBINS, GATE, ref['t'] / t_chosen))

## Sweep 2 (secondary, COUPLED): global l_max_ncdm at the chosen q_size

l_max_ncdm is global (precisions.h:312) - lowering it also lowers the neutrino hierarchy accuracy, so the high-l C_l tail (dTail) is the binding metric here, not just the daughter.

In [ ]:
LMAX_SWEEP = [17, 12, 8, 6]
lrows = []
for lm in LMAX_SWEEP:
    c = ref if (CHOSEN_NBINS == REF_NBINS and lm == REF_LMAX) else run_case(CHOSEN_NBINS, lm)
    row = dict(l_max=lm, t=c['t'], speedup=ref['t'] / c['t'],
               dpk=dev_pk(c, ref), dcl=dev_cl(c, ref), dtail=dev_cl(c, ref, ell_lo=1500))
    lrows.append(row)
    print('l_max={:>3}  q_size={}  {:6.1f}s  {:5.2f}x  dPk={:.2e}  dCl={:.2e}  dTail={:.2e}'.format(
        lm, CHOSEN_NBINS, row['t'], row['speedup'], row['dpk'], row['dcl'], row['dtail']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)\nlm = [r['l_max'] for r in lrows]\nax[0].semilogy(lm, [r['dpk'] for r in lrows], 'o-', color=qual[0], label='P(k)')\nax[0].semilogy(lm, [r['dcl'] for r in lrows], 's-', color=qual[1], label='C_l TT')\nax[0].semilogy(lm, [r['dtail'] for r in lrows], '^-', color=qual[2], label='C_l tail (l>=1500)')\nax[0].axhline(TOL_PK, color='k', ls='--', lw=1)\nax[0].set_xlabel('l_max_ncdm (global)'); ax[0].set_ylabel('max |deviation| vs full-res')\nax[0].set_title('convergence vs l_max'); ax[0].legend(loc='best'); ax[0].grid(True, which='both', alpha=0.3)\nax[1].plot(lm, [r['speedup'] for r in lrows], 'o-', color=qual[2])\nax[1].axhline(1.0, color='k', ls=':', lw=1)\nax[1].set_xlabel('l_max_ncdm (global)'); ax[1].set_ylabel('speedup (t_ref / t)')\nax[1].set_title('speedup vs l_max'); ax[1].grid(True, which='both', alpha=0.3)\nplt.show()\n\ndef lmax_ok(r):\n    return (r['dpk'] < TOL_PK and r['dcl'] < TOL_CL and r['dtail'] < TOL_TAIL)\nlok = [r for r in lrows if lmax_ok(r)]\nCHOSEN_LMAX = min((r['l_max'] for r in lok), default=REF_LMAX)\nfinal_speedup = ref['t'] / next(r['t'] for r in lrows if r['l_max'] == CHOSEN_LMAX)\nprint()\nprint('RECOMMENDATION (gate {:.0e} on P(k), C_l, tail vs full-res reference):'.format(TOL_PK))\nprint('  ncdm_N_momentum_bins = 15, {}    (q_size sweep -> the real lever)'.format(CHOSEN_NBINS))\nprint('  l_max_ncdm           = {}    (P(k) is binding, not the neutrino tail; reducing trades ~3.5% P(k) for <2.2x)'.format(CHOSEN_LMAX))\nprint('  measured speedup     = {:.2f}x  ({:.0f}s -> {:.0f}s)'.format(\n    final_speedup, ref['t'], ref['t'] / final_speedup))\nif CHOSEN_NBINS == REF_NBINS and CHOSEN_LMAX == REF_LMAX:\n    print('  VERDICT: no reduction within tolerance -> daughter is irreducibly expensive at the exact level.')\nelse:\n    print('  VERDICT: adopt the reduced q_size as default config - safe speedup, exact hierarchy + physics untouched.')

## Findings / caveats\n\n- **q_size is the win; l_max is not.** `q_size = 1001` is ~8x overkill: q ~ 120-250 gives <=1% P(k) / <=0.4% C_l at ~10x / ~4.7x. Lowering the global `l_max_ncdm` instead costs ~3.5% P(k) at l_max=12 for only ~1.45x - a bad trade. Keep `l_max_ncdm = 17`.\n- **P(k) is the binding metric for l_max** (not the neutrino C_l tail, which stays fine down to l_max~8). Likely the daughter's own free-streaming needing the multipoles; if a follow-up shows the NEUTRINO drives it, a per-species `l_max_ncdm_acc` could let the daughter cutoff drop - but the upside is <2.2x vs the q_size ~10x, so probably not worth it.\n- **dW is a diagnostic, not a gate.** Its large values are a cold-tail artifact (w->0 at late a inflates a relative metric on a negligible absolute difference); the gate is on P(k)/C_l/tail. dev_w is now restricted to w>1%.\n- **Convergence is to the 1001-bin run, not truth.** If a deviation looks suspiciously flat, add a point ABOVE 1001 to confirm the reference is itself converged.\n- **The l_max sweep ran at CHOSEN_NBINS.** With the fixed picker that is the reduced q_size; re-run res09 -> res12 for a self-consistent combined recommendation (cheap - the reduced-q l_max runs are fast).\n- No rebuild needed: `ncdm_N_momentum_bins` and `l_max_ncdm` are runtime parameters against the built classy.